# Graph/Network Anomaly Detection

This notebook demonstrates quantum anomaly detection on network intrusion data
using the KDD Cup 99 dataset. We compare three quantum methods against classical
baselines, including a Level B comparison of QAOA vs Simulated Annealing.

**Dataset**: KDD Cup 99 — network connections labeled as normal or attack.

**Quantum methods**: Quantum Kernel + One-Class SVM, QAOA Clustering, Quantum Distance k-NN.

**Classical baselines**: Isolation Forest, One-Class SVM, LOF, DBSCAN, Elliptic Envelope,
Simulated Annealing.

In [ ]:
# ============================================================
# Configuration — change these values to experiment
# ============================================================
SEED = 42
N_QUBITS = 8
N_SAMPLES = 2000
CONTAMINATION = 0.1  # KDD has higher anomaly ratio
N_KERNEL_SAMPLES = 120
N_QAOA_POINTS = 12
CIRCUIT_SCALE = 0.2  # Scale factor for circuit drawings


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from quantum_anomaly_detection.visualization.plots import (
    plot_optimization_history,
    plot_2d_scatter,
    plot_roc_curves,
    plot_kernel_matrix,
)
from quantum_anomaly_detection.evaluation.metrics import compute_metrics, format_results_table


## 1. Data Loading & Exploration

The KDD Cup 99 dataset contains network connection records with features like
duration, protocol type, bytes transferred, etc. Each connection is labeled as
normal or as a specific attack type. We treat all attacks as anomalies.

In [ ]:
from quantum_anomaly_detection.data.graph import (
    load_kdd_cup, preprocess_graph_features, build_adjacency_from_features
)

X_raw, y = load_kdd_cup(subsample=N_SAMPLES, seed=SEED)
print(f'Dataset: {len(X_raw)} connections, {X_raw.shape[1]} features')
print(f'Normal: {(y==0).sum()}, Attack: {(y==1).sum()}')
print(f'Attack ratio: {y.mean():.2%}')

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(6, 3))
labels_count = ['Normal', 'Attack']
counts = [(y==0).sum(), (y==1).sum()]
ax.bar(labels_count, counts, color=['steelblue', 'salmon'])
ax.set_ylabel('Count')
ax.set_title('KDD Cup 99 — Class Distribution')
for i, c in enumerate(counts):
    ax.text(i, c + 20, str(c), ha='center')
plt.tight_layout()
plt.show()

## 2. Preprocessing

Categorical features are label-encoded, then all features pass through StandardScaler
+ PCA to N_QUBITS dimensions, scaled to [0, pi] for quantum encoding.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=SEED, stratify=y
)

X_train = preprocess_graph_features(X_train_raw, n_components=N_QUBITS, fit_data=X_train_raw)
X_test = preprocess_graph_features(X_test_raw, n_components=N_QUBITS, fit_data=X_train_raw)

X_train_normal = X_train[y_train == 0]
print(f'Train: {len(X_train)} ({(y_train==0).sum()} normal), Test: {len(X_test)}')
print(f'Feature shape: {X_train.shape[1]}, range [{X_train.min():.2f}, {X_train.max():.2f}]')

## 3. Quantum Kernel + One-Class SVM

The quantum kernel encodes network connection features into quantum states. The ZZ
feature map creates entanglement between feature pairs, enabling the kernel to detect
non-linear patterns that distinguish normal traffic from attacks.

In [ ]:
from quantum_anomaly_detection.circuits.feature_maps import build_zz_feature_map
from quantum_anomaly_detection.methods.quantum_kernel import quantum_kernel_svm, compute_kernel_matrix
from quantum_anomaly_detection.visualization.plots import plot_kernel_matrix

feature_map_zz = build_zz_feature_map(N_QUBITS, reps=1)
print(f'ZZ Feature Map: {N_QUBITS} qubits')
feature_map_zz.draw('mpl', fold=20, scale=CIRCUIT_SCALE)

In [ ]:
rng = np.random.default_rng(SEED)
n_sub = min(N_KERNEL_SAMPLES, len(X_train_normal))
idx_train = rng.choice(len(X_train_normal), n_sub, replace=False)
idx_test_k = rng.choice(len(X_test), min(N_KERNEL_SAMPLES, len(X_test)), replace=False)

preds_qk, scores_qk, _ = quantum_kernel_svm(
    X_train_normal[idx_train], X_test[idx_test_k], feature_map_zz, nu=CONTAMINATION
)

K = compute_kernel_matrix(X_train_normal[idx_train[:25]], feature_map_zz)
plot_kernel_matrix(K, title='Quantum Kernel Matrix (KDD Features)')
plt.show()

In [ ]:
from quantum_anomaly_detection.evaluation.metrics import compute_metrics

results = {}
m = compute_metrics(y_test[idx_test_k], preds_qk, scores_qk)
results['Quantum Kernel SVM'] = m
print('Quantum Kernel One-Class SVM:', {k: f'{v:.3f}' for k, v in m.items()})

## 4. QAOA Clustering

QAOA solves a combinatorial optimization problem: partition data points into 2 clusters
by minimizing intra-cluster distance. The QUBO formulation uses one qubit per data point
with cost H = sum D[i,j]/2 * (I - Z_i Z_j). Small or isolated clusters are flagged
as anomalies.

Due to qubit limitations, we subsample to N_QAOA_POINTS points.

In [ ]:
from quantum_anomaly_detection.methods.qaoa_clustering import (
    run_qaoa_clustering, identify_anomaly_clusters
)
from quantum_anomaly_detection.circuits.qaoa import build_clustering_hamiltonian, build_qaoa_circuit
from quantum_anomaly_detection.visualization.plots import plot_optimization_history, plot_2d_scatter

# Subsample for QAOA
# Ensure subsample includes some anomalies
idx_anomaly_test = np.where(y_test == 1)[0]
idx_normal_test = np.where(y_test == 0)[0]
n_anom = min(3, len(idx_anomaly_test))
n_norm = N_QAOA_POINTS - n_anom
idx_qaoa = np.concatenate([
    rng.choice(idx_normal_test, n_norm, replace=False),
    rng.choice(idx_anomaly_test, n_anom, replace=False),
])
X_qaoa = X_test[idx_qaoa]
y_qaoa = y_test[idx_qaoa]

print(f'QAOA subsample: {len(X_qaoa)} points ({(y_qaoa==1).sum()} attacks)')

In [ ]:
# Visualize QAOA circuit
from scipy.spatial.distance import pdist, squareform
D_qaoa = squareform(pdist(X_qaoa))
D_qaoa_norm = D_qaoa / D_qaoa.max()

H_cluster = build_clustering_hamiltonian(D_qaoa_norm)
qaoa_circ = build_qaoa_circuit(H_cluster, reps=2)
print(f'QAOA circuit: {qaoa_circ.num_qubits} qubits, {qaoa_circ.num_parameters} parameters')

In [ ]:
# Run QAOA clustering
labels_qaoa, history_qaoa = run_qaoa_clustering(
    X_qaoa, reps=2, maxiter=200, seed=SEED
)

plot_optimization_history(history_qaoa, title='QAOA Clustering Optimization')
plt.show()

anomalies_qaoa = identify_anomaly_clusters(labels_qaoa, X_qaoa, min_cluster_fraction=0.2)
m = compute_metrics(y_qaoa, anomalies_qaoa)
results['QAOA Clustering (subsample)'] = m
print('QAOA Clustering:', {k: f'{v:.3f}' for k, v in m.items()})
print(f'Cluster sizes: {(labels_qaoa==0).sum()} vs {(labels_qaoa==1).sum()}')

## 5. Quantum Distance k-NN

Quantum distance estimation measures the fidelity between feature-encoded quantum states.
Network connections that are distant from their neighbors in quantum feature space are
likely anomalies (attacks).

In [ ]:
from quantum_anomaly_detection.circuits.feature_maps import build_angle_encoding_map
from quantum_anomaly_detection.methods.quantum_distance import detect_anomalies_knn

fm_angle = build_angle_encoding_map(N_QUBITS)
fm_angle.draw('mpl', fold=20, scale=CIRCUIT_SCALE)

In [ ]:
from quantum_anomaly_detection.methods.vqc_autoencoder import score_anomalies, detect_anomalies

n_dist = min(80, len(X_test))
idx_dist = rng.choice(len(X_test), n_dist, replace=False)

preds_qdist, scores_qdist = detect_anomalies_knn(
    X_test[idx_dist], fm_angle, k=5, contamination=CONTAMINATION
)

m = compute_metrics(y_test[idx_dist], preds_qdist, scores_qdist)
results['Quantum Distance k-NN'] = m
print('Quantum Distance k-NN:', {k: f'{v:.3f}' for k, v in m.items()})

## 6. Classical Benchmarks

### Level A: Classical Anomaly Detection

In [ ]:
from quantum_anomaly_detection.classical.benchmarks import (
    run_isolation_forest, run_svm, run_lof, run_dbscan, run_elliptic_envelope
)

preds_if, scores_if = run_isolation_forest(X_train_normal, X_test, CONTAMINATION, SEED)
results['Isolation Forest'] = compute_metrics(y_test, preds_if, scores_if)

preds_ocsvm, scores_ocsvm = run_svm(X_train_normal, X_test, nu=CONTAMINATION)
results['One-Class SVM (RBF)'] = compute_metrics(y_test, preds_ocsvm, scores_ocsvm)

preds_lof, scores_lof = run_lof(X_train_normal, X_test, contamination=CONTAMINATION)
results['LOF'] = compute_metrics(y_test, preds_lof, scores_lof)

preds_ee, scores_ee = run_elliptic_envelope(X_train_normal, X_test, CONTAMINATION)
results['Elliptic Envelope'] = compute_metrics(y_test, preds_ee, scores_ee)

# DBSCAN
dbscan_labels = run_dbscan(X_test, eps=1.5, min_samples=5)
preds_dbscan = np.where(dbscan_labels == -1, 1, 0)  # -1 = noise = anomaly
results['DBSCAN'] = compute_metrics(y_test, preds_dbscan)

print('Classical methods computed.')

### Level B: QAOA vs Simulated Annealing

We solve the same QUBO clustering problem using Simulated Annealing and compare
the result against QAOA. Both methods operate on the same small subsample.

In [ ]:
from quantum_anomaly_detection.classical.benchmarks import run_simulated_annealing

# SA on same distance matrix
labels_sa, history_sa = run_simulated_annealing(D_qaoa_norm, n_iter=5000, seed=SEED)
anomalies_sa = identify_anomaly_clusters(labels_sa, X_qaoa, min_cluster_fraction=0.2)

m_sa = compute_metrics(y_qaoa, anomalies_sa)
results['Simulated Annealing (subsample)'] = m_sa

print(f'QAOA clusters:  {labels_qaoa} -> anomalies: {anomalies_qaoa}')
print(f'SA clusters:    {labels_sa} -> anomalies: {anomalies_sa}')
print(f'True labels:    {y_qaoa.astype(int)}')

In [ ]:
# Compare optimization histories
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history_qaoa, color='steelblue', linewidth=1)
ax1.set_title('QAOA Optimization')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Cost')

ax2.plot(history_sa, color='coral', linewidth=0.5)
ax2.set_title('Simulated Annealing')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Cost')

plt.tight_layout()
plt.show()

## 7. Comparison

### Results Table

In [ ]:
from quantum_anomaly_detection.evaluation.metrics import format_results_table

df = format_results_table(results)
print(df.to_string())

In [ ]:
from quantum_anomaly_detection.visualization.plots import plot_roc_curves

roc_data = {
    'Quantum Kernel': (y_test[idx_test_k], scores_qk),
    'Quantum Distance': (y_test[idx_dist], scores_qdist),
    'Isolation Forest': (y_test, scores_if),
    'One-Class SVM (RBF)': (y_test, scores_ocsvm),
    'LOF': (y_test, scores_lof),
}

plot_roc_curves(roc_data, title='ROC Curves — Network Intrusion Detection')
plt.show()

In [ ]:
# 2D visualization of test data with true labels vs predictions
plot_2d_scatter(X_test, y_test, title='True Labels', method_name='Ground Truth')
plt.show()

## Discussion

- **Quantum Kernel One-Class SVM** captures complex feature interactions through entanglement
  in the ZZ feature map. The kernel matrix reveals cluster structure in the data.
- **QAOA Clustering** partitions a small subset into clusters and identifies the
  minority cluster as anomalous. Limited by qubit count but demonstrates quantum
  optimization for anomaly detection.
- **Level B comparison**: QAOA and Simulated Annealing solve the same QUBO. SA
  typically converges more reliably on small instances, but QAOA demonstrates the
  quantum approach to combinatorial optimization.
- **Quantum Distance k-NN** provides geometric anomaly detection using quantum state
  fidelity as a distance metric.
- Classical methods (especially Isolation Forest) remain strong baselines for network
  intrusion detection at scale. Quantum methods currently operate on subsamples but
  demonstrate feasibility for future larger-scale quantum hardware.